# 01 · Tokenizers: BPE vs unigram

Team 1 owns the tokenizer, but everyone should understand it. Here we train two small
SentencePiece models (BPE and unigram) on a text sample and compare how efficiently they encode text.

The class tokenizer is trained the same way with `vocab_size=32000` on ~2 GB of FineWeb-Edu text:
```
python -m gw1b.prepare_data sample-text --sample 10BT --out $GW1B_SCRATCH/data/tokenizer_corpus.txt --mb 2000
python -m gw1b.tokenizer train --input $GW1B_SCRATCH/data/tokenizer_corpus.txt --out $GW1B_GROUP/tokenizer/gw1b-32k --vocab 32000 --type bpe
```

In [ ]:
# --- GW1B setup cell (same in every notebook) -------------------------------------------------
import os, sys, json, time, pathlib
GW1B_HOME    = os.environ.get("GW1B_HOME", "/SEAS/groups/gw1b/gw1b-f2026")
GW1B_SCRATCH = os.environ.get("GW1B_SCRATCH", "/scratch/gw1b-class")
GW1B_GROUP   = os.environ.get("GW1B_GROUP", "/SEAS/groups/gw1b")
USER_DIR     = os.path.join(GW1B_SCRATCH, "users", os.environ.get("USER", "student"))
os.makedirs(USER_DIR, exist_ok=True)
if GW1B_HOME not in sys.path:
    sys.path.insert(0, GW1B_HOME)        # the shared gw1b package
import jax, jax.numpy as jnp, numpy as np
print("JAX", jax.__version__, "| backend:", jax.default_backend(), "| devices:", jax.devices())
print("your scratch dir:", USER_DIR)


## 1. A text sample
Use real FineWeb-Edu text if the class corpus is on scratch, otherwise a small synthetic corpus.

In [ ]:
from gw1b.prepare_data import parquet_files, iter_documents
sample_path = os.path.join(USER_DIR, "tokenizer_sample.txt")
try:
    files = parquet_files("10BT")[:1]
    with open(sample_path, "w") as f:
        for i, doc in enumerate(iter_documents(files)):
            f.write(doc.replace("\n", " ") + "\n")
            if i >= 20000: break
    print("using FineWeb-Edu text from", files[0])
except FileNotFoundError:
    import random
    random.seed(0)
    # a synthetic "language": 6000 pseudo-words built from syllables, Zipf-distributed like real text
    syll = [c + v for c in "bcdfghjklmnprstvwz" for v in ["a", "e", "i", "o", "u", "ai", "ou"]]
    words = ["".join(random.choices(syll, k=random.randint(1, 4))) for _ in range(6000)]
    weights = [1.0 / (i + 1) for i in range(len(words))]
    with open(sample_path, "w") as f:
        for _ in range(20000):
            f.write(" ".join(random.choices(words, weights=weights, k=random.randint(8, 30))) + ".\n")
    print("FineWeb-Edu not found on scratch; using a synthetic corpus")
print(open(sample_path).read(300))

## 2. Train BPE vs unigram

In [ ]:
from gw1b.tokenizer import train_sentencepiece, Tokenizer
toks = {}
for kind in ["bpe", "unigram"]:
    path = train_sentencepiece(sample_path, os.path.join(USER_DIR, f"tok-{kind}-4k"), vocab_size=4000,
                               model_type=kind, input_sentence_size=20000)
    toks[kind] = Tokenizer(path)
    print(kind, "vocab:", toks[kind].vocab_size, "special ids (unk,bos,eos,pad):",
          toks[kind].unk_id, toks[kind].bos_id, toks[kind].eos_id, toks[kind].pad_id)

## 3. Compare compression (bytes per token) — the metric that decides how many tokens your 100B-token budget really buys

In [ ]:
text = open(sample_path).read(200_000)
for kind, tok in toks.items():
    s = tok.stats(text)
    print(f"{kind:8s} tokens={s['tokens']:>7,d}  bytes/token={s['bytes_per_token']:.2f}  words/token={s['words_per_token']:.2f}")
print()
sentence = "Pegasus trains the GW1B language model with JAX on eight A100 GPUs."
for kind, tok in toks.items():
    print(f"{kind:8s}", tok.pieces(sentence))

## 4. Questions Team 1 can answer with this
* 16K vs 32K vs 50K vocabulary: bytes/token vs. embedding size (vocab × d_model parameters!)
* BPE vs unigram on code / math / non-English text
* `byte_fallback=True` means no `<unk>`: every byte is representable. Try `tok.pieces("日本語 ∑ 🎓")`.

Next: `02_data_pipeline.ipynb`.